In [4]:
import pandas as pd
import numpy as np

def evaluate_alpha_theory():
    # --- 1. Calculate dP/dT from NVT data ---
    print("--- Step 1: Extracting dP/dT ---")
    nvt_files = {
        0.000010: "nvt_lj_T0.000010_g0.dat",
        0.000015: "nvt_lj_T0.000015_g0.dat"
    }
    
    pressures = {}
    for T, filename in nvt_files.items():
        try:
            df = pd.read_csv(filename, sep=r'\s+', header=None, names=['Time', 'KE', 'PE', 'Pressure'])
            pressures[T] = df.iloc[len(df)//2:]['Pressure'].mean() # Discard warmup
        except FileNotFoundError:
            print(f"Error: {filename} not found.")
            return

    T1, T2 = 0.000010, 0.000015
    dp_dt = (pressures[T2] - pressures[T1]) / (T2 - T1)
    print(f"dP/dT = {dp_dt:.8f}")

    # --- 2. Extract Bulk Modulus (K) from Elasticity file ---
    print("\n--- Step 2: Extracting Bulk Modulus (K) ---")
    elasticity_file = "3dlj_elasticity_N4000_s000.dat"
    
    try:
        # Load the raw text data into a numpy array
        elastic_data = np.loadtxt(elasticity_file)
        print(f"Raw elasticity data: {elastic_data}")
        
        # IMPORTANT: You must point this index to the column that represents K.
        # Often, C-code elasticity outputs are formatted like: 
        # [Glass_Serial, Density, Pressure, Bulk_Modulus (K), Shear_Modulus (G)]
        # We assume K is at index 3 (the 4th value). Change this if necessary!
        if elastic_data.ndim == 1:
            K = elastic_data[3]  
        else:
            K = elastic_data[0, 4] # If multiple rows, take the first row
            
        print(f"Extracted K = {K:.8f}")
        
    except Exception as e:
        print(f"Error reading {elasticity_file}: {e}")
        return

    # --- 3. Final Alpha Calculation ---
    print("\n--- Step 3: Final Alpha Metric ---")
    alpha = dp_dt / K
    
    print(f"Alpha (thermal expansivity) = {alpha:.8f}")

# Run the complete pipeline
evaluate_alpha_theory()

--- Step 1: Extracting dP/dT ---
dP/dT = 4.34975570

--- Step 2: Extracting Bulk Modulus (K) ---
Raw elasticity data: [[ 0.00000000e+00 -5.84364001e+00 -5.25706325e-03  6.54571135e+00
   2.51394763e+01  5.54969138e-04  3.53949765e+01]
 [ 1.00000000e+00 -5.85446873e+00 -2.97244460e-02  5.88890856e+00
   2.49310385e+01  2.21942086e-01  3.41516629e+01]
 [ 2.00000000e+00 -5.85678995e+00 -5.39687386e-02  6.44974410e+00
   2.48755482e+01 -1.04352912e-01  2.96043127e+01]
 [ 3.00000000e+00 -5.84966048e+00 -3.05401561e-02  6.39476691e+00
   2.49816829e+01  2.23725784e-01  3.33694283e+01]
 [ 4.00000000e+00 -5.84284472e+00 -1.93717457e-02  6.29758277e+00
   2.51709005e+01  8.97129054e-02  4.14313502e+01]
 [ 5.00000000e+00 -5.84272598e+00 -5.40490143e-03  6.42531229e+00
   2.51305162e+01  1.87871545e-01  3.54671417e+01]
 [ 6.00000000e+00 -5.83924862e+00  1.02398813e-02  6.32128761e+00
   2.52391734e+01 -2.13364446e-01  3.09180708e+01]
 [ 7.00000000e+00 -5.84763393e+00 -8.40187974e-03  6.33859787e+

In [3]:
"""
alpha_sanity_check.py
=====================
Sanity check for finite-temperature alpha = (1/K) * dP_vir/dT at const V.

What this script does
---------------------
  1. Loads all available NVT files for a given glass (default: glass 0)
  2. Loads K from the elasticity output file
  3. Computes alpha for EVERY combinatorial subset of >= 3 temperature points
  4. Computes alpha for random subsets of fixed size
  5. Runs rolling-window fits over the sorted T axis
  6. Bootstraps the time-series of each temperature point to estimate
     how much of the alpha scatter comes from equilibration noise vs.
     genuine nonlinearity

All diagnostics are saved to a multi-panel PNG.

Usage
-----
  python alpha_sanity_check.py
  python alpha_sanity_check.py --glass 0 --data_dir my_data --n_random 5000

Folder layout expected
----------------------
  <DATA_DIR>/
    3dlj_elasticity_N4000_s000.dat      (elasticity output)
    nvt_lj_T{T:.6f}_g{glass}.dat       (NVT time-series for each T)
"""

import argparse
import itertools
import warnings
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr

# ─────────────────────────────── constants ────────────────────────────────── #

TEMPERATURES_ALL = np.array([
    0.000001, 0.000005, 0.000010, 0.000015, 0.000020,
    0.000025, 0.000030, 0.000035, 0.000040,
])

SKIP_FRACTION = 0.30   # discard first 30% as equilibration
N_ATOMS       = 4000
RHO           = 0.541
MIN_SUBSET    = 3      # smallest subset size for combinatorial analysis

# ──────────────────────────────── I/O helpers ─────────────────────────────── #

def load_elasticity(data_dir: Path, glass: int):
    """Return K for the requested glass (and all K values for context)."""
    epath = data_dir / "3dlj_elasticity_N4000_s000.dat"
    if not epath.exists():
        raise FileNotFoundError(f"Elasticity file not found: {epath}")
    elas = np.loadtxt(epath, comments="#")
    # columns: serial  u_per_N  pressure  G  K  first_order  second_order
    serials = elas[:, 0].astype(int)
    K_all   = elas[:, 4]
    K_dict  = {int(serials[i]): K_all[i] for i in range(len(serials))}
    if glass not in K_dict:
        raise KeyError(f"Glass {glass} not found in elasticity file (available: {list(K_dict)})")
    return K_dict[glass], K_dict


def load_nvt_file(filepath: Path, skip: float = SKIP_FRACTION):
    """
    Load one NVT file.  Columns: time  KE/N  PE/N  P_virial.

    Returns
    -------
    T_meas  : float   – measured temperature  (2/3) * <KE/N>
    P_mean  : float   – mean virial pressure over production window
    P_err   : float   – standard error of P in production window
    P_series: ndarray – full production-window pressure time series
    """
    d      = np.loadtxt(filepath)
    n_skip = max(1, int(len(d) * skip))
    prod   = d[n_skip:]
    T_meas = (2.0 / 3.0) * prod[:, 1].mean()
    P_mean = prod[:, 3].mean()
    P_err  = prod[:, 3].std() / np.sqrt(len(prod))
    return T_meas, P_mean, P_err, prod[:, 3]


def load_all_nvt(data_dir: Path, glass: int, temperatures=TEMPERATURES_ALL):
    """
    Load all available NVT files for one glass.

    Returns
    -------
    T_meas  : (n,) array of measured temperatures
    P_mean  : (n,) array of mean pressures
    P_err   : (n,) array of pressure standard errors
    P_series: list of n 1-D arrays (production pressure time-series)
    T_target: (n,) array of target temperatures (for labelling)
    """
    T_meas_list  = []
    P_mean_list  = []
    P_err_list   = []
    P_series_list = []
    T_target_list = []

    for T in sorted(temperatures):
        fpath = data_dir / f"nvt_lj_T{T:.6f}_g{glass}.dat"
        if not fpath.exists():
            continue
        Tm, Pm, Pe, Ps = load_nvt_file(fpath)
        T_meas_list.append(Tm)
        P_mean_list.append(Pm)
        P_err_list.append(Pe)
        P_series_list.append(Ps)
        T_target_list.append(T)

    if len(T_meas_list) < 2:
        raise RuntimeError(
            f"Need at least 2 NVT files for glass {glass} in {data_dir}.\n"
            f"Found: {len(T_meas_list)}"
        )

    return (np.array(T_meas_list), np.array(P_mean_list),
            np.array(P_err_list), P_series_list, np.array(T_target_list))


# ─────────────────────────── fitting utilities ────────────────────────────── #

def wls_slope(T_vals, P_vals, P_errs=None):
    """
    Weighted (or unweighted) linear fit  P = a + b*T.

    Returns (slope b, slope_stderr).
    """
    T = np.asarray(T_vals, dtype=float)
    P = np.asarray(P_vals, dtype=float)
    n = len(T)

    if P_errs is None or np.all(np.asarray(P_errs) == 0):
        w = np.ones(n)
    else:
        e = np.asarray(P_errs, dtype=float)
        e = np.where(e > 0, e, e[e > 0].min() * 1e-3)  # guard zero errors
        w = 1.0 / e**2

    # solve normal equations  [sum(w)  sum(wT) ; sum(wT)  sum(wT^2)] [a;b] = [sum(wP); sum(wTP)]
    sw   = w.sum()
    swT  = (w * T).sum()
    swT2 = (w * T**2).sum()
    swP  = (w * P).sum()
    swTP = (w * T * P).sum()

    det = sw * swT2 - swT**2
    if abs(det) < 1e-30:
        return np.nan, np.nan

    b = (sw * swTP - swT * swP) / det
    # variance of b
    var_b = sw / det
    return b, np.sqrt(var_b)


def bootstrap_slope(T_vals, P_series_list, n_boot=2000, rng=None):
    """
    Block-bootstrap dP/dT by resampling production-window frames independently
    per temperature.  Returns array of n_boot slope estimates.
    """
    if rng is None:
        rng = np.random.default_rng(42)
    T = np.asarray(T_vals, dtype=float)
    slopes = []
    for _ in range(n_boot):
        P_boot = np.array([
            rng.choice(ps, size=len(ps), replace=True).mean()
            for ps in P_series_list
        ])
        # unweighted OLS for speed (errors are effectively re-estimated each boot)
        b, _ = wls_slope(T, P_boot)
        if np.isfinite(b):
            slopes.append(b)
    return np.array(slopes)


# ─────────────────────────── subset analysis ──────────────────────────────── #

def all_subset_alphas(T_meas, P_mean, P_err, K, min_size=MIN_SUBSET):
    """
    Enumerate every subset of size >= min_size.
    Returns list of dicts with keys: indices, T_min, T_max, T_range, n_pts, alpha, slope_err.
    """
    n = len(T_meas)
    results = []
    for size in range(min_size, n + 1):
        for idx in itertools.combinations(range(n), size):
            idx = list(idx)
            T_sub = T_meas[idx]
            P_sub = P_mean[idx]
            E_sub = P_err[idx]
            b, be = wls_slope(T_sub, P_sub, E_sub)
            if not np.isfinite(b):
                continue
            results.append({
                "indices"   : idx,
                "n_pts"     : size,
                "T_min"     : T_sub.min(),
                "T_max"     : T_sub.max(),
                "T_range"   : T_sub.max() - T_sub.min(),
                "T_mean"    : T_sub.mean(),
                "slope"     : b,
                "slope_err" : be,
                "alpha"     : b / K,
                "alpha_err" : be / K,
            })
    return results


def random_subset_alphas(T_meas, P_mean, P_err, K, subset_size,
                          n_random=3000, seed=0):
    """
    Draw n_random random subsets of a fixed size and compute alpha for each.
    """
    rng = np.random.default_rng(seed)
    n   = len(T_meas)
    alphas = []
    for _ in range(n_random):
        idx = sorted(rng.choice(n, size=subset_size, replace=False))
        b, _ = wls_slope(T_meas[idx], P_mean[idx], P_err[idx])
        if np.isfinite(b):
            alphas.append(b / K)
    return np.array(alphas)


def rolling_window_alphas(T_meas, P_mean, P_err, K, window=3):
    """
    Compute alpha for every consecutive window of `window` temperature points.
    Returns arrays (T_center, alpha_window).
    """
    n = len(T_meas)
    if window > n:
        return np.array([]), np.array([])
    T_centers = []
    alphas    = []
    for i in range(n - window + 1):
        T_sub = T_meas[i:i+window]
        P_sub = P_mean[i:i+window]
        E_sub = P_err[i:i+window]
        b, _ = wls_slope(T_sub, P_sub, E_sub)
        if np.isfinite(b):
            T_centers.append(T_sub.mean())
            alphas.append(b / K)
    return np.array(T_centers), np.array(alphas)


# ──────────────────────────────── plotting ────────────────────────────────── #

GOLD   = "#E6A817"
STEEL  = "#2E6FA3"
CORAL  = "#D45F4B"
SAGE   = "#5BA85A"
PURPLE = "#7B5EA7"
GREY   = "#7A7A7A"

def make_figure(T_meas, P_mean, P_err, P_series_list,
                K, K_glass, glass, data_dir,
                all_subsets, n_random=3000, n_boot=1500):

    alpha_full, _ = wls_slope(T_meas, P_mean, P_err)
    alpha_full /= K

    # ── random subsets by size ──────────────────────────────────────── #
    n = len(T_meas)
    sizes_to_probe = list(range(MIN_SUBSET, n + 1))
    rand_by_size   = {
        sz: random_subset_alphas(T_meas, P_mean, P_err, K,
                                  subset_size=sz, n_random=n_random)
        for sz in sizes_to_probe
    }

    # ── rolling windows ─────────────────────────────────────────────── #
    roll3_Tc, roll3_a = rolling_window_alphas(T_meas, P_mean, P_err, K, window=3)
    roll4_Tc, roll4_a = rolling_window_alphas(T_meas, P_mean, P_err, K, window=4)
    roll5_Tc, roll5_a = rolling_window_alphas(T_meas, P_mean, P_err, K, window=5)

    # ── bootstrap dP/dT ─────────────────────────────────────────────── #
    boot_slopes = bootstrap_slope(T_meas, P_series_list, n_boot=n_boot)
    boot_alphas = boot_slopes / K

    # ── all-subsets arrays ───────────────────────────────────────────── #
    sub_alpha  = np.array([s["alpha"]   for s in all_subsets])
    sub_n      = np.array([s["n_pts"]   for s in all_subsets])
    sub_tmax   = np.array([s["T_max"]   for s in all_subsets])
    sub_tmin   = np.array([s["T_min"]   for s in all_subsets])
    sub_trange = np.array([s["T_range"] for s in all_subsets])

    # ── figure layout ────────────────────────────────────────────────── #
    fig = plt.figure(figsize=(20, 22))
    fig.patch.set_facecolor("#F8F8F8")
    gs = gridspec.GridSpec(4, 3, figure=fig,
                           hspace=0.50, wspace=0.38,
                           top=0.94, bottom=0.04,
                           left=0.07, right=0.97)

    title_str = (
        f"Alpha Sanity Check — Glass {glass}  |  "
        f"K = {K:.5f}  |  "
        f"n_T = {n}  |  "
        f"Total subsets analysed: {len(all_subsets)}"
    )
    fig.suptitle(title_str, fontsize=13, fontweight="bold", color="#1A1A2E")

    def style_ax(ax, title, xlabel, ylabel):
        ax.set_title(title, fontsize=9, fontweight="bold", pad=6, color="#1A1A2E")
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.25, lw=0.6)
        ax.set_facecolor("#FDFDFD")
        for sp in ax.spines.values():
            sp.set_linewidth(0.7)

    def vline_full(ax):
        ax.axhline(alpha_full, color=CORAL, lw=1.6, ls="--",
                   label=f"full-set α = {alpha_full:.4e}", zorder=5)

    # ── Panel 1: raw P(T) + fit ───────────────────────────────────────── #
    ax1 = fig.add_subplot(gs[0, :2])
    T_line = np.linspace(0, T_meas.max() * 1.15, 200)
    b_full, _ = wls_slope(T_meas, P_mean, P_err)
    a_full  = P_mean.mean() - b_full * T_meas.mean()
    ax1.errorbar(T_meas * 1e5, P_mean, yerr=P_err,
                 fmt="o", color=STEEL, ms=7, capsize=4, lw=1.4,
                 ecolor=STEEL, alpha=0.85, label="NVT data  ± SEM", zorder=4)
    ax1.plot(T_line * 1e5, a_full + b_full * T_line, "--",
             color=CORAL, lw=2, label=f"Full-set WLS: slope={b_full:.4f}")
    ax1.set_xlabel("Temperature T  (×10⁻⁵ LJ units)", fontsize=9)
    ax1.set_ylabel("⟨P_virial⟩", fontsize=9)
    ax1.set_title(
        f"Panel 1 — P_virial vs T  (glass {glass})\n"
        f"Full-set α = {alpha_full:.4e}     K = {K:.5f}",
        fontsize=9, fontweight="bold", pad=6, color="#1A1A2E",
    )
    ax1.legend(fontsize=8, framealpha=0.8)
    ax1.grid(alpha=0.25, lw=0.6); ax1.set_facecolor("#FDFDFD")
    for sp in ax1.spines.values(): sp.set_linewidth(0.7)

    # residuals inset
    ax1r = ax1.inset_axes([0.70, 0.12, 0.28, 0.38])
    resid = P_mean - (a_full + b_full * T_meas)
    ax1r.axhline(0, color="black", lw=0.8)
    ax1r.bar(range(n), resid / abs(P_mean).mean() * 100,
             color=[STEEL if r > 0 else CORAL for r in resid],
             edgecolor="none", alpha=0.8)
    ax1r.set_xlabel("T index", fontsize=6)
    ax1r.set_ylabel("resid (%)", fontsize=6)
    ax1r.set_title("Residuals", fontsize=6.5)
    ax1r.tick_params(labelsize=5)
    ax1r.grid(alpha=0.2)

    # ── Panel 2: histogram of all-subset alphas ───────────────────────── #
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(sub_alpha, bins=40, color=STEEL, alpha=0.75, edgecolor="white", lw=0.4)
    vline_full(ax2)
    ax2.axvline(np.median(sub_alpha), color=GOLD, lw=1.6, ls=":",
                label=f"median = {np.median(sub_alpha):.4e}")
    cv = np.std(sub_alpha) / np.abs(np.mean(sub_alpha)) * 100
    style_ax(ax2,
             f"Panel 2 — α distribution: ALL subsets (n≥{MIN_SUBSET})\n"
             f"CV = {cv:.1f}%   range = [{sub_alpha.min():.3e}, {sub_alpha.max():.3e}]",
             "α = dP/dT / K", "Count")
    ax2.legend(fontsize=7, framealpha=0.8)

    # ── Panel 3: α vs T_max of subset ────────────────────────────────── #
    ax3 = fig.add_subplot(gs[1, 0])
    sc3 = ax3.scatter(sub_tmax * 1e5, sub_alpha, c=sub_n,
                      cmap="viridis_r", s=16, alpha=0.55, zorder=3)
    vline_full(ax3)
    plt.colorbar(sc3, ax=ax3, label="# T points", pad=0.01)
    style_ax(ax3,
             "Panel 3 — α vs T_max of subset\n"
             "(high T_max → anharmonic contamination?)",
             "T_max  (×10⁻⁵)", "α")
    ax3.legend(fontsize=7)
    ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}"))

    # ── Panel 4: α vs T_min of subset ────────────────────────────────── #
    ax4 = fig.add_subplot(gs[1, 1])
    sc4 = ax4.scatter(sub_tmin * 1e5, sub_alpha, c=sub_n,
                      cmap="viridis_r", s=16, alpha=0.55, zorder=3)
    vline_full(ax4)
    plt.colorbar(sc4, ax=ax4, label="# T points", pad=0.01)
    style_ax(ax4,
             "Panel 4 — α vs T_min of subset\n"
             "(high T_min → missing low-T anchor?)",
             "T_min  (×10⁻⁵)", "α")
    ax4.legend(fontsize=7)

    # ── Panel 5: α vs T_range of subset ──────────────────────────────── #
    ax5 = fig.add_subplot(gs[1, 2])
    sc5 = ax5.scatter(sub_trange * 1e5, sub_alpha, c=sub_n,
                      cmap="viridis_r", s=16, alpha=0.55, zorder=3)
    vline_full(ax5)
    plt.colorbar(sc5, ax=ax5, label="# T points", pad=0.01)

    # correlation
    rho, pval = pearsonr(sub_trange, sub_alpha)
    style_ax(ax5,
             f"Panel 5 — α vs T_range of subset\n"
             f"Pearson r = {rho:.3f}  (p = {pval:.3g})\n"
             "(wider range = more stable estimate if P(T) is linear)",
             "T_range  (×10⁻⁵)", "α")
    ax5.legend(fontsize=7)

    # ── Panel 6: α spread vs subset size (box) ───────────────────────── #
    ax6 = fig.add_subplot(gs[2, 0])
    box_data   = [rand_by_size[sz] for sz in sizes_to_probe]
    box_labels = [str(sz) for sz in sizes_to_probe]
    bp = ax6.boxplot(box_data, labels=box_labels,
                     patch_artist=True, notch=False,
                     medianprops=dict(color=CORAL, lw=2),
                     whiskerprops=dict(lw=1),
                     flierprops=dict(marker=".", ms=3, alpha=0.3))
    for patch in bp["boxes"]:
        patch.set(facecolor=STEEL, alpha=0.55)
    ax6.axhline(alpha_full, color=CORAL, lw=1.5, ls="--",
                label=f"full-set α = {alpha_full:.4e}")

    cvs = [np.std(rand_by_size[sz]) / abs(np.mean(rand_by_size[sz])) * 100
           for sz in sizes_to_probe]
    ax6_twin = ax6.twinx()
    ax6_twin.plot(range(1, len(sizes_to_probe)+1), cvs, "^--",
                  color=GOLD, ms=7, lw=1.5, label="CV (%)")
    ax6_twin.set_ylabel("CV (%)", fontsize=8, color=GOLD)
    ax6_twin.tick_params(axis="y", labelcolor=GOLD, labelsize=7)

    style_ax(ax6,
             f"Panel 6 — α spread vs subset size\n"
             f"({n_random} random subsets each; CV = coefficient of variation)",
             "# Temperature points in subset", "α")
    ax6.legend(fontsize=7, loc="upper left")

    # ── Panel 7: rolling-window local slopes ─────────────────────────── #
    ax7 = fig.add_subplot(gs[2, 1])
    if len(roll3_Tc):
        ax7.plot(roll3_Tc * 1e5, roll3_a, "o-", color=STEEL,
                 ms=7, lw=1.5, label="window=3")
    if len(roll4_Tc):
        ax7.plot(roll4_Tc * 1e5, roll4_a, "s-", color=SAGE,
                 ms=7, lw=1.5, label="window=4")
    if len(roll5_Tc):
        ax7.plot(roll5_Tc * 1e5, roll5_a, "^-", color=PURPLE,
                 ms=7, lw=1.5, label="window=5")
    vline_full(ax7)
    ax7.axhline(0, color="black", lw=0.7, ls=":")
    style_ax(ax7,
             "Panel 7 — Rolling-window local α vs T_center\n"
             "(flat → linear P(T); sloped → anharmonic curvature)",
             "T_center  (×10⁻⁵)", "α  (local window fit)")
    ax7.legend(fontsize=7)

    # ── Panel 8: bootstrap α distribution ────────────────────────────── #
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.hist(boot_alphas, bins=50, color=SAGE, alpha=0.7,
             edgecolor="white", lw=0.4, density=True)
    ax8.axvline(np.mean(boot_alphas), color=CORAL, lw=2, ls="--",
                label=f"mean = {np.mean(boot_alphas):.4e}")
    ax8.axvline(np.percentile(boot_alphas, 2.5), color=GREY, lw=1, ls=":")
    ax8.axvline(np.percentile(boot_alphas, 97.5), color=GREY, lw=1, ls=":",
                label=f"95% CI [{np.percentile(boot_alphas,2.5):.3e}, "
                      f"{np.percentile(boot_alphas,97.5):.3e}]")
    ci_width = np.percentile(boot_alphas, 97.5) - np.percentile(boot_alphas, 2.5)
    cv_boot  = np.std(boot_alphas) / abs(np.mean(boot_alphas)) * 100
    style_ax(ax8,
             f"Panel 8 — Bootstrap α (n_boot = {n_boot})\n"
             f"Uncertainty from equilibration noise alone\n"
             f"CV = {cv_boot:.2f}%   95% CI width = {ci_width:.3e}",
             "α", "Density")
    ax8.legend(fontsize=7, framealpha=0.8)

    # ── Panel 9: α vs index of excluded temperature point ─────────────── #
    ax9 = fig.add_subplot(gs[3, 0])
    leave_one_out = []
    for i in range(n):
        idx  = [j for j in range(n) if j != i]
        b, _ = wls_slope(T_meas[idx], P_mean[idx], P_err[idx])
        leave_one_out.append(b / K)
    leave_one_out = np.array(leave_one_out)
    colors_loo = [CORAL if abs(a - alpha_full) / abs(alpha_full) > 0.05 else STEEL
                  for a in leave_one_out]
    ax9.bar(range(n), leave_one_out, color=colors_loo, alpha=0.8, edgecolor="white")
    ax9.axhline(alpha_full, color=GOLD, lw=1.8, ls="--",
                label=f"full-set α = {alpha_full:.4e}")
    ax9.set_xticks(range(n))
    ax9.set_xticklabels([f"excl\nT{i+1}" for i in range(n)], fontsize=6.5)
    pct_change = (leave_one_out - alpha_full) / abs(alpha_full) * 100
    for i, pc in enumerate(pct_change):
        ax9.text(i, leave_one_out[i] + np.sign(pc) * abs(leave_one_out).max() * 0.015,
                 f"{pc:+.1f}%", ha="center", va="bottom", fontsize=5.5,
                 color="#333333")
    style_ax(ax9,
             "Panel 9 — Leave-one-out: α when each T point excluded\n"
             "(red bar → that point shifts α by > 5%; T points numbered by rank)",
             "Excluded temperature point", "α")
    ax9.legend(fontsize=7)

    # ── Panel 10: α scatter coloured by T_min and T_max ──────────────── #
    ax10 = fig.add_subplot(gs[3, 1])
    # Separate subsets by whether they include the lowest T point
    t_low      = T_meas.min()
    has_low    = np.array([T_meas[s["indices"][0]] == t_low for s in all_subsets])
    ax10.scatter(sub_trange[~has_low] * 1e5, sub_alpha[~has_low],
                 s=12, alpha=0.4, color=GREY, label="excludes lowest T", zorder=2)
    ax10.scatter(sub_trange[has_low] * 1e5, sub_alpha[has_low],
                 s=12, alpha=0.55, color=STEEL, label="includes lowest T", zorder=3)
    vline_full(ax10)
    mean_with    = np.mean(sub_alpha[has_low])
    mean_without = np.mean(sub_alpha[~has_low])
    ax10.axhline(mean_with,    color=STEEL, lw=1, ls=":",
                 label=f"mean (w/ T_low)   = {mean_with:.4e}")
    ax10.axhline(mean_without, color=GREY,  lw=1, ls=":",
                 label=f"mean (w/o T_low)  = {mean_without:.4e}")
    style_ax(ax10,
             f"Panel 10 — Does including the lowest-T point matter?\n"
             f"Mean shift = {(mean_with - mean_without) / abs(alpha_full) * 100:+.1f}% of full α",
             "T_range  (×10⁻⁵)", "α")
    ax10.legend(fontsize=6, framealpha=0.9, ncol=1)

    # ── Panel 11: numeric summary text ───────────────────────────────── #
    ax11 = fig.add_subplot(gs[3, 2])
    ax11.axis("off")
    ax11.set_facecolor("#F0F4FF")

    summary_lines = [
        ("SUMMARY — Glass %d" % glass, True),
        ("", False),
        ("K (bulk modulus)         %9.5f" % K, False),
        ("n_T points loaded         %5d" % n, False),
        ("Total subsets (n≥%d)    %6d" % (MIN_SUBSET, len(all_subsets)), False),
        ("", False),
        ("Full-set WLS alpha        %+.4e" % alpha_full, False),
        ("", False),
        ("Alpha statistics (all subsets):", True),
        ("  mean                     %+.4e" % np.mean(sub_alpha), False),
        ("  std                       %.4e" % np.std(sub_alpha), False),
        ("  CV                        %6.2f%%" % (np.std(sub_alpha)/abs(np.mean(sub_alpha))*100), False),
        ("  min                      %+.4e" % sub_alpha.min(), False),
        ("  max                      %+.4e" % sub_alpha.max(), False),
        ("", False),
        ("Bootstrap (equil. noise):", True),
        ("  CV_boot                   %6.2f%%" % cv_boot, False),
        ("  95%% CI width              %.3e" % ci_width, False),
        ("", False),
        ("Rolling window local α:", True),
    ]
    if len(roll3_a):
        summary_lines += [
            ("  w=3  range [%.3e, %.3e]" % (roll3_a.min(), roll3_a.max()), False),
            ("  w=3  CV    %6.2f%%" % (np.std(roll3_a)/abs(np.mean(roll3_a))*100), False),
        ]
    summary_lines += [
        ("", False),
        ("Leave-one-out max shift:", True),
        ("  max |Δα/α|              %6.2f%%" % np.max(abs(pct_change)), False),
        ("  point index             %s (T rank)" % str(np.argmax(abs(pct_change))+1), False),
        ("", False),
        ("DIAGNOSTIC:", True),
        ("  subset CV >> boot CV    → noise from subset choice", False),
        ("  rolling α not flat      → P(T) nonlinear (anharmonic)", False),
        ("  LOO shift > 10%         → outlier T point exists", False),
    ]

    y0 = 0.97
    dy = 0.047
    for (line, bold) in summary_lines:
        weight = "bold" if bold else "normal"
        color  = "#1A1A2E" if bold else "#333333"
        ax11.text(0.04, y0, line,
                  transform=ax11.transAxes,
                  fontsize=7.2, family="monospace",
                  fontweight=weight, color=color, va="top")
        y0 -= dy

    for spine in ax11.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color("#BBBBCC")

    return fig


# ────────────────────────────── main ─────────────────────────────────────── #

def main():
    parser = argparse.ArgumentParser(
        description="Alpha sanity-check: subset sensitivity for finite-T alpha."
    )
    parser.add_argument("--glass",    type=int,   default=0,
                        help="Glass index to analyse (default: 0)")
    parser.add_argument("--data_dir", type=str,   default="initial_files",
                        help="Directory with NVT and elasticity output files")
    parser.add_argument("--n_random", type=int,   default=3000,
                        help="Number of random subsets per size (default: 3000)")
    parser.add_argument("--n_boot",   type=int,   default=1500,
                        help="Bootstrap resamples (default: 1500)")
    parser.add_argument("--out",      type=str,   default="alpha_sanity.png",
                        help="Output figure filename")
    parser.add_argument("--skip",     type=float, default=SKIP_FRACTION,
                        help="Fraction of NVT run to discard as equilibration")
    args = parser.parse_args(args=["--data_dir", "."])

    data_dir = Path(args.data_dir)
    glass    = args.glass

    print(f"=== Alpha Sanity Check — Glass {glass} ===")
    print(f"Data directory : {data_dir.resolve()}")

    # ── load K ────────────────────────────────────────────────────────── #
    K, K_all = load_elasticity(data_dir, glass)
    print(f"Bulk modulus K = {K:.6f}  (glass {glass})")

    # ── load NVT data ─────────────────────────────────────────────────── #
    print("Loading NVT files...")
    T_meas, P_mean, P_err, P_series_list, T_target = load_all_nvt(
        data_dir, glass
    )
    n = len(T_meas)
    print(f"  Loaded {n} temperature points:")
    print(f"  {'T_target':>12}  {'T_measured':>12}  {'P_mean':>12}  {'P_SEM':>12}")
    print(f"  {'-'*52}")
    for i in range(n):
        print(f"  {T_target[i]:>12.6f}  {T_meas[i]:>12.6f}  "
              f"{P_mean[i]:>12.8f}  {P_err[i]:>12.2e}")

    # ── all subsets ───────────────────────────────────────────────────── #
    total = sum(
        len(list(itertools.combinations(range(n), k)))
        for k in range(MIN_SUBSET, n + 1)
    )
    print(f"\nEnumerating all {total} subsets of size >= {MIN_SUBSET}...")
    all_subsets = all_subset_alphas(T_meas, P_mean, P_err, K)
    print(f"  Done.  alpha range: [{min(s['alpha'] for s in all_subsets):.4e}, "
          f"{max(s['alpha'] for s in all_subsets):.4e}]")

    sub_alpha = np.array([s["alpha"] for s in all_subsets])
    cv = np.std(sub_alpha) / abs(np.mean(sub_alpha)) * 100
    alpha_full, _ = wls_slope(T_meas, P_mean, P_err)
    alpha_full /= K
    print(f"  Full-set alpha = {alpha_full:.6e}")
    print(f"  All-subset CV  = {cv:.2f}%")

    # ── rolling windows ────────────────────────────────────────────────── #
    print("\nRolling window slopes:")
    for w in [3, 4, 5]:
        Tc, al = rolling_window_alphas(T_meas, P_mean, P_err, K, window=w)
        if len(al):
            print(f"  window={w}: alpha range [{al.min():.4e}, {al.max():.4e}]  "
                  f"CV={np.std(al)/abs(np.mean(al))*100:.2f}%")

    # ── generate figure ────────────────────────────────────────────────── #
    print(f"\nBuilding figure ({args.n_random} random subsets / size, "
          f"{args.n_boot} bootstrap samples)...")
    fig = make_figure(
        T_meas, P_mean, P_err, P_series_list,
        K, K_all, glass, data_dir,
        all_subsets,
        n_random=args.n_random,
        n_boot=args.n_boot,
    )

    out_path = Path(args.out)
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#F8F8F8")
    plt.close(fig)
    print(f"\nFigure saved → {out_path.resolve()}")

    # ── print diagnostic verdict ─────────────────────────────────────── #
    print("\n" + "="*60)
    print("DIAGNOSTIC VERDICT")
    print("="*60)

    boot_slopes = bootstrap_slope(T_meas, P_series_list, n_boot=500)
    boot_alphas = boot_slopes / K
    cv_boot = np.std(boot_alphas) / abs(np.mean(boot_alphas)) * 100

    roll3_Tc, roll3_a = rolling_window_alphas(T_meas, P_mean, P_err, K, window=3)

    print(f"  Subset-choice CV      = {cv:.2f}%")
    print(f"  Bootstrap (equil.) CV = {cv_boot:.2f}%")
    if cv > 3 * cv_boot:
        print("  ⚠ Subset choice dominates variability.")
        print("    → alpha is SENSITIVE to which T points you include.")
        print("    → P(T) is likely nonlinear over this T range,")
        print("      or some T points are poorly equilibrated.")
    else:
        print("  ✓ Subset choice and equilibration noise are comparable.")

    if len(roll3_a) >= 2:
        roll3_cv = np.std(roll3_a) / abs(np.mean(roll3_a)) * 100
        print(f"\n  Rolling-window (w=3) local α CV = {roll3_cv:.2f}%")
        if roll3_cv > 5.0:
            print("  ⚠ Local slope changes significantly across T range.")
            print("    → P(T) is curved — anharmonic regime may be entered.")
        else:
            print("  ✓ Local slope is roughly constant (P(T) approximately linear).")

    leave_one_out = []
    for i in range(n):
        idx  = [j for j in range(n) if j != i]
        b, _ = wls_slope(T_meas[idx], P_mean[idx], P_err[idx])
        leave_one_out.append(b / K)
    loo_pct = abs((np.array(leave_one_out) - alpha_full) / alpha_full * 100)
    worst_i = np.argmax(loo_pct)
    print(f"\n  Leave-one-out: excluding T{worst_i+1} (T = {T_meas[worst_i]:.6f})")
    print(f"  shifts α by {loo_pct[worst_i]:+.2f}%")
    if loo_pct[worst_i] > 10:
        print(f"  ⚠ That temperature point has outsized leverage.")
    else:
        print(f"  ✓ No single temperature point dominates the fit.")

    print("="*60)


if __name__ == "__main__":
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        main()

=== Alpha Sanity Check — Glass 0 ===
Data directory : /home/sangam/vsc/glass_simulations/LJ_glass/initial_files
Bulk modulus K = 25.139476  (glass 0)
Loading NVT files...
  Loaded 9 temperature points:
      T_target    T_measured        P_mean         P_SEM
  ----------------------------------------------------
      0.000001      0.000001   -0.00524919      6.49e-07
      0.000005      0.000005   -0.00523610      1.40e-06
      0.000010      0.000010   -0.00521828      1.95e-06
      0.000015      0.000015   -0.00519685      2.47e-06
      0.000020      0.000020   -0.00517533      2.80e-06
      0.000025      0.000025   -0.00514844      3.19e-06
      0.000030      0.000030   -0.00513618      3.49e-06
      0.000035      0.000035   -0.00511247      3.81e-06
      0.000040      0.000040   -0.00509213      4.04e-06

Enumerating all 466 subsets of size >= 3...
  Done.  alpha range: [1.3535e-01, 1.9083e-01]
  Full-set alpha = 1.585592e-01
  All-subset CV  = 3.62%

Rolling window slopes:


/tmp/ipykernel_1007/4181229887.py:421: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax6.boxplot(box_data, labels=box_labels,



Figure saved → /home/sangam/vsc/glass_simulations/LJ_glass/initial_files/alpha_sanity.png

DIAGNOSTIC VERDICT
  Subset-choice CV      = 3.62%
  Bootstrap (equil.) CV = 1.94%
  ✓ Subset choice and equilibration noise are comparable.

  Rolling-window (w=3) local α CV = 11.36%
  ⚠ Local slope changes significantly across T range.
    → P(T) is curved — anharmonic regime may be entered.

  Leave-one-out: excluding T1 (T = 0.000001)
  shifts α by +4.15%
  ✓ No single temperature point dominates the fit.
